<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Humidity/Humidity_Online_learning_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install river --quiet

In [2]:
import pandas as pd
import numpy as np

from river import metrics, compose, preprocessing

print("River imported successfully")

River imported successfully


In [3]:
path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/final_humidity_dataset (1).csv"
humidity_df = pd.read_csv(path)

print("Dataset shape:", humidity_df.shape)
humidity_df.head()

Dataset shape: (130003, 16)


,uv_index,cloud,condition_text,air_quality_Ozone,temperature_celsius,feels_like_celsius,air_quality_us-epa-index,air_quality_gb-defra-index,longitude,precip_mm,air_quality_PM10,air_quality_PM2.5,visibility_km,air_quality_Sulphur_dioxide,latitude,humidity
0,1.0,0,2,62.2,16.1,16.1,1,1,-120.49,0.00,7.1,6.3,16.0,0.2,46.60,58
1,1.0,37,32,23.3,23.0,25.3,2,2,-87.22,0.28,25.3,19.0,10.0,1.4,14.10,78
2,1.0,50,23,5.9,26.0,30.2,2,2,-89.20,0.30,28.1,20.4,10.0,7.5,13.71,94
3,1.0,100,19,0.4,20.0,20.0,4,10,-90.53,0.09,178.1,132.0,5.0,19.3,14.62,88
4,1.0,94,30,34.0,26.0,29.6,1,1,-88.77,0.00,32.1,7.7,10.0,0.2,17.25,89


In [4]:
target = "humidity"

In [5]:
# -----------------------------
# FEATURE ENGINEERING FOR HUMIDITY
# -----------------------------
humidity_df["humidity_lag1"] = humidity_df[target].shift(1)
humidity_df["humidity_lag2"] = humidity_df[target].shift(2)
humidity_df["humidity_lag3"] = humidity_df[target].shift(3)

humidity_df["humidity_roll3_mean"] = humidity_df[target].rolling(window=3).mean()
humidity_df["humidity_roll5_mean"] = humidity_df[target].rolling(window=5).mean()

# -----------------------------
# RELATED FEATURES (from selected humidity features)
# -----------------------------
if "temperature_celsius" in humidity_df.columns:
    humidity_df["temp_lag1"] = humidity_df["temperature_celsius"].shift(1)

if "feels_like_celsius" in humidity_df.columns:
    humidity_df["feels_like_lag1"] = humidity_df["feels_like_celsius"].shift(1)

if "cloud" in humidity_df.columns:
    humidity_df["cloud_lag1"] = humidity_df["cloud"].shift(1)

if "precip_mm" in humidity_df.columns:
    humidity_df["precip_lag1"] = humidity_df["precip_mm"].shift(1)

if "visibility_km" in humidity_df.columns:
    humidity_df["visibility_lag1"] = humidity_df["visibility_km"].shift(1)

if "air_quality_Ozone" in humidity_df.columns:
    humidity_df["ozone_lag1"] = humidity_df["air_quality_Ozone"].shift(1)

if "air_quality_PM10" in humidity_df.columns:
    humidity_df["pm10_lag1"] = humidity_df["air_quality_PM10"].shift(1)

if "air_quality_PM2.5" in humidity_df.columns:
    humidity_df["pm25_lag1"] = humidity_df["air_quality_PM2.5"].shift(1)

if "air_quality_Sulphur_dioxide" in humidity_df.columns:
    humidity_df["so2_lag1"] = humidity_df["air_quality_Sulphur_dioxide"].shift(1)

# -----------------------------
# CLEAN NA VALUES
# -----------------------------
humidity_df = humidity_df.dropna().reset_index(drop=True)

print("After feature engineering:", humidity_df.shape)
humidity_df.head()

After feature engineering: (129999, 30)


,uv_index,cloud,condition_text,air_quality_Ozone,temperature_celsius,feels_like_celsius,air_quality_us-epa-index,air_quality_gb-defra-index,longitude,precip_mm,...,humidity_roll5_mean,temp_lag1,feels_like_lag1,cloud_lag1,precip_lag1,visibility_lag1,ozone_lag1,pm10_lag1,pm25_lag1,so2_lag1
0,1.0,94,30,34.0,26.0,29.6,1,1,-88.77,0.00,...,81.4,20.0,20.0,100.0,0.09,5.0,0.4,178.1,132.0,19.3
1,1.0,75,41,14.3,27.2,30.6,1,1,-86.27,0.01,...,85.8,26.0,29.6,94.0,0.00,10.0,34.0,32.1,7.7,0.2
2,1.0,75,4,0.0,21.0,21.0,2,2,-84.08,0.17,...,90.2,27.2,30.6,75.0,0.01,10.0,14.3,14.7,11.7,11.4
3,1.0,5,2,7.4,20.8,20.8,2,3,-99.13,0.00,...,80.8,21.0,21.0,75.0,0.17,7.0,0.0,23.3,21.7,6.6
4,1.0,84,41,0.2,21.9,21.9,1,1,-76.75,0.10,...,80.0,20.8,20.8,5.0,0.00,10.0,7.4,48.0,35.1,16.9


In [7]:
selected_features = [col for col in humidity_df.columns if col != target]

print("Number of features:", len(selected_features))
print(selected_features)

Number of features: 29
['uv_index', 'cloud', 'condition_text', 'air_quality_Ozone', 'temperature_celsius', 'feels_like_celsius', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'longitude', 'precip_mm', 'air_quality_PM10', 'air_quality_PM2.5', 'visibility_km', 'air_quality_Sulphur_dioxide', 'latitude', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'humidity_roll3_mean', 'humidity_roll5_mean', 'temp_lag1', 'feels_like_lag1', 'cloud_lag1', 'precip_lag1', 'visibility_lag1', 'ozone_lag1', 'pm10_lag1', 'pm25_lag1', 'so2_lag1']


In [8]:
split_index = int(0.8 * len(humidity_df))

train_df = humidity_df.iloc[:split_index].copy()
test_df = humidity_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (103999, 30)
Test shape : (26000, 30)


In [9]:
from river import linear_model

online_model = compose.Pipeline(
    preprocessing.StandardScaler(),
    linear_model.LinearRegression()
)

model_name = "River Online Only"
print("Using model:", model_name)

Using model: River Online Only


In [10]:
train_mse = metrics.MSE()
train_rmse = metrics.RMSE()
train_mae = metrics.MAE()
train_r2 = metrics.R2()

train_true = []
train_pred = []

for _, row in train_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    # predict current instance
    y_pred = online_model.predict_one(x)
    if y_pred is None:
        y_pred = 0.0

    train_true.append(y)
    train_pred.append(y_pred)

    train_mse.update(y, y_pred)
    train_rmse.update(y, y_pred)
    train_mae.update(y, y_pred)
    train_r2.update(y, y_pred)

    # learn from the same instance
    online_model.learn_one(x, y)

print("\nOnline Learning - Training Stream Results")
print("MSE :", round(train_mse.get(), 4))
print("RMSE:", round(train_rmse.get(), 4))
print("MAE :", round(train_mae.get(), 4))
print("R2  :", round(train_r2.get(), 4))
print("Accuracy (%):", round(train_r2.get() * 100, 2))


Online Learning - Training Stream Results
MSE : 8.027687079668439e+21
RMSE: 89597360896.7833
MAE : 16974200329.5929
R2  : -1.376166301824372e+19
Accuracy (%): -1.376166301824372e+21


In [11]:
online_train_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(train_mse.get(), 3),
    "RMSE": round(train_rmse.get(), 3),
    "MAE": round(train_mae.get(), 3),
    "R2": round(train_r2.get(), 3),
    "Accuracy (%)": round(train_r2.get() * 100, 3)
}])

online_train_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River Online Only,8.027687e+21,8.959736e+10,1.697420e+10,-1.376166e+19,-1.376166e+21


In [12]:
test_mse = metrics.MSE()
test_rmse = metrics.RMSE()
test_mae = metrics.MAE()
test_r2 = metrics.R2()

test_true = []
test_pred = []

for _, row in test_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    y_hat = online_model.predict_one(x)
    if y_hat is None:
        y_hat = 0.0

    test_true.append(y)
    test_pred.append(y_hat)

    test_mse.update(y, y_hat)
    test_rmse.update(y, y_hat)
    test_mae.update(y, y_hat)
    test_r2.update(y, y_hat)

print("\nOnline Learning - Final Test Results")
print("MSE :", round(test_mse.get(), 4))
print("RMSE:", round(test_rmse.get(), 4))
print("MAE :", round(test_mae.get(), 4))
print("R2  :", round(test_r2.get(), 4))
print("Accuracy (%):", round(test_r2.get() * 100, 2))


Online Learning - Final Test Results
MSE : 9878072556746250.0
RMSE: 99388493.0802
MAE : 74967000.6724
R2  : -20610270900575.44
Accuracy (%): -2061027090057544.2


In [13]:
online_test_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(test_mse.get(), 3),
    "RMSE": round(test_rmse.get(), 3),
    "MAE": round(test_mae.get(), 3),
    "R2": round(test_r2.get(), 3),
    "Accuracy (%)": round(test_r2.get() * 100, 3)
}])

online_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River Online Only,9.878073e+15,99388493.08,7.496700e+07,-2.061027e+13,-2.061027e+15


In [14]:
from google.colab import files

online_train_df.to_csv("online_only_humidity_training_results.csv", index=False)
online_test_df.to_csv("online_only_humidity_test_results.csv", index=False)

files.download("online_only_humidity_training_results.csv")
files.download("online_only_humidity_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>